## Fase 6C: Construção do Modelo de IA para Previsão de Produtividade (Parte 1)

Neste notebook, vamos iniciar a construção do modelo de IA para previsão de produtividade agrícola, utilizando as features extraídas nas fases anteriores.

In [ ]:
# Configuração do ambiente
import sys
sys.path.append('../../')
import setup_notebook
setup_notebook.setup_environment()

%matplotlib inline

## Introdução

Nas fases anteriores, realizamos o pré-processamento dos dados de NDVI/EVI e produtividade agrícola (Fase 6A) e a extração de informações relevantes para o modelo de IA (Fase 6B). Agora, vamos construir o modelo de IA para previsão de produtividade agrícola.

Neste notebook (Parte 1), vamos focar nas seguintes tarefas:

1. Carregamento das features selecionadas
2. Exploração e preparação dos dados para modelagem
3. Seleção e avaliação de diferentes algoritmos de aprendizado de máquina
4. Treinamento inicial dos modelos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
import warnings

# Configurar o estilo dos gráficos
plt.style.use('fivethirtyeight')
sns.set(style="whitegrid")

# Ignorar avisos
warnings.filterwarnings('ignore')

## 1. Carregamento das Features Selecionadas

Vamos carregar as features selecionadas na Fase 6B.

In [ ]:
# Verificar se os arquivos existem
if os.path.exists('../../sprint2/assets/features_relevantes.csv'):
    # Carregar as features relevantes
    df_features = pd.read_csv('../../sprint2/assets/features_relevantes.csv')
    print("Features relevantes carregadas com sucesso!")
else:
    # Se o arquivo não existir, carregar as features finais
    if os.path.exists('../../sprint2/assets/features_finais.csv'):
        df_features = pd.read_csv('../../sprint2/assets/features_finais.csv')
        print("Features finais carregadas com sucesso!")
    else:
        # Se nenhum arquivo existir, carregar os dados originais e processá-los novamente
        print("Os arquivos de features não foram encontrados. Carregando os dados originais...")
        
        # Carregar os dados de NDVI/EVI
        df_ndvi_nf = pd.read_csv('../../assets/ndvi_mensal_nova_friburgo.csv')
        df_ndvi_t = pd.read_csv('../../assets/ndvi_mensal_teresopolis.csv')
        
        # Carregar os dados de produtividade agrícola
        df_prod_combinados = pd.read_csv('../../assets/dados_produtividade_combinados.csv')
        
        # Filtrar os dados de produtividade para o ano de 2017
        df_prod_2017 = df_prod_combinados[df_prod_combinados['Ano'] == 2017]
        
        # Agrupar os dados de produtividade por município
        df_prod_mean = df_prod_2017.groupby('Município')['Produtividade (t/ha)'].mean().reset_index()
        
        # Filtrar os dados de NDVI/EVI para o ano de 2017
        df_ndvi_nf_2017 = df_ndvi_nf[df_ndvi_nf['Ano'] == 2017]
        df_ndvi_t_2017 = df_ndvi_t[df_ndvi_t['Ano'] == 2017]
        
        # Calcular a média anual do EVI para cada município em 2017
        evi_nf_2017 = df_ndvi_nf_2017['EVI_mean'].mean()
        evi_t_2017 = df_ndvi_t_2017['EVI_mean'].mean()
        
        # Criar um dataframe com os dados de EVI e produtividade
        data = {
            'Município': ['Nova Friburgo', 'Teresópolis'],
            'EVI_mean': [evi_nf_2017, evi_t_2017]
        }
        df_features = pd.DataFrame(data)
        
        # Mesclar com os dados de produtividade
        df_features = pd.merge(df_features, df_prod_mean, on='Município')
        
        print("Dados carregados e processados manualmente.")

# Exibir informações sobre o dataframe
print("\nInformações sobre o dataframe de features:")
print(f"Número de registros: {df_features.shape[0]}")
print(f"Número de colunas: {df_features.shape[1]}")
print(f"Colunas: {', '.join(df_features.columns)}")

# Exibir o dataframe
print("\nDataframe de features:")
print(df_features)

## 2. Exploração e Preparação dos Dados para Modelagem

Vamos explorar os dados e prepará-los para a modelagem.

In [ ]:
# Verificar se há valores ausentes
print("Valores ausentes no dataframe:")
print(df_features.isnull().sum())

# Estatísticas descritivas
print("\nEstatísticas descritivas:")
print(df_features.describe())

# Matriz de correlação
print("\nMatriz de correlação:")
corr_matrix = df_features.drop(columns=['Município']).corr()
print(corr_matrix)

In [ ]:
# Visualizar a matriz de correlação
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlação das Features')
plt.tight_layout()
plt.show()

In [ ]:
# Preparar os dados para modelagem
# Separar as features (X) e o target (y)
X = df_features.drop(columns=['Município', 'Produtividade (t/ha)'])
y = df_features['Produtividade (t/ha)']

# Exibir as features e o target
print("Features (X):")
print(X)
print("\nTarget (y):")
print(y)

## 3. Seleção e Avaliação de Diferentes Algoritmos de Aprendizado de Máquina

Vamos selecionar e avaliar diferentes algoritmos de aprendizado de máquina para a previsão de produtividade agrícola.

In [ ]:
# Definir os modelos a serem avaliados
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'ElasticNet': ElasticNet(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'SVR': SVR()
}

# Avaliar os modelos usando validação cruzada
results = {}
for name, model in models.items():
    # Criar um pipeline com StandardScaler e o modelo
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    
    # Treinar o modelo com todos os dados
    pipeline.fit(X, y)
    
    # Fazer previsões
    y_pred = pipeline.predict(X)
    
    # Calcular métricas
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)
    
    # Armazenar os resultados
    results[name] = {
        'RMSE': rmse,
        'R²': r2
    }

# Exibir os resultados
results_df = pd.DataFrame(results).T
print("Resultados da avaliação dos modelos:")
print(results_df.sort_values('RMSE'))

In [ ]:
# Visualizar os resultados
plt.figure(figsize=(12, 6))

# Ordenar os resultados pelo RMSE
results_df_sorted = results_df.sort_values('RMSE')

# Plotar o RMSE
plt.subplot(1, 2, 1)
results_df_sorted['RMSE'].plot(kind='bar')
plt.title('RMSE por Modelo')
plt.xlabel('Modelo')
plt.ylabel('RMSE')
plt.xticks(rotation=45, ha='right')
plt.grid(True)

# Plotar o R²
plt.subplot(1, 2, 2)
results_df_sorted['R²'].plot(kind='bar')
plt.title('R² por Modelo')
plt.xlabel('Modelo')
plt.ylabel('R²')
plt.xticks(rotation=45, ha='right')
plt.grid(True)

plt.tight_layout()
plt.show()

## 4. Treinamento Inicial dos Modelos

Vamos treinar os modelos com melhor desempenho na avaliação anterior.

In [ ]:
# Selecionar os 3 melhores modelos com base no RMSE
best_models = results_df.sort_values('RMSE').head(3).index.tolist()
print(f"Melhores modelos: {best_models}")

# Treinar os melhores modelos
trained_models = {}
for name in best_models:
    # Criar um pipeline com StandardScaler e o modelo
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', models[name])
    ])
    
    # Treinar o modelo com todos os dados
    pipeline.fit(X, y)
    
    # Fazer previsões
    y_pred = pipeline.predict(X)
    
    # Calcular métricas
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    
    # Armazenar o modelo treinado e as métricas
    trained_models[name] = {
        'pipeline': pipeline,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'y_pred': y_pred
    }

# Exibir as métricas dos modelos treinados
metrics = {name: {'RMSE': model['RMSE'], 'MAE': model['MAE'], 'R²': model['R²']} 
          for name, model in trained_models.items()}
metrics_df = pd.DataFrame(metrics).T
print("\nMétricas dos modelos treinados:")
print(metrics_df)

In [ ]:
# Visualizar as previsões dos modelos
plt.figure(figsize=(12, 6))

# Plotar os valores reais
plt.scatter(range(len(y)), y, color='blue', label='Valores Reais', s=100)

# Plotar as previsões de cada modelo
for i, (name, model) in enumerate(trained_models.items()):
    plt.scatter(range(len(y)), model['y_pred'], marker='x', s=100, label=f'Previsões - {name}')

plt.title('Valores Reais vs. Previsões')
plt.xlabel('Índice')
plt.ylabel('Produtividade (t/ha)')
plt.xticks(range(len(y)), df_features['Município'])
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Exportação dos Modelos Treinados

Vamos exportar os modelos treinados para serem utilizados na próxima parte.

In [ ]:
# Importar joblib para salvar os modelos
import joblib

# Criar diretório para os modelos
os.makedirs('../../sprint2/models', exist_ok=True)

# Salvar os modelos treinados
for name, model in trained_models.items():
    # Salvar o pipeline (scaler + modelo)
    joblib.dump(model['pipeline'], f'../../sprint2/models/{name.replace(" ", "_").lower()}_pipeline.pkl')
    
    # Salvar as métricas
    metrics = {
        'RMSE': model['RMSE'],
        'MAE': model['MAE'],
        'R²': model['R²']
    }
    pd.DataFrame([metrics]).to_csv(f'../../sprint2/models/{name.replace(" ", "_").lower()}_metrics.csv', index=False)

print("Modelos exportados com sucesso!")

## Conclusão da Parte 1

Neste notebook, iniciamos a construção do modelo de IA para previsão de produtividade agrícola. Carregamos as features selecionadas na Fase 6B, exploramos e preparamos os dados para modelagem, selecionamos e avaliamos diferentes algoritmos de aprendizado de máquina, e treinamos os modelos com melhor desempenho.

Principais observações:

1. Avaliamos 7 algoritmos de aprendizado de máquina diferentes: Regressão Linear, Ridge, Lasso, ElasticNet, Random Forest, Gradient Boosting e SVR.
2. Identificamos os 3 melhores modelos com base no RMSE: [lista dos 3 melhores modelos].
3. Treinamos os melhores modelos e calculamos as métricas de desempenho (RMSE, MAE e R²).
4. Exportamos os modelos treinados para serem utilizados na próxima parte.

Na próxima parte (Fase 6C - Parte 2), vamos otimizar os hiperparâmetros dos melhores modelos, realizar uma análise mais detalhada dos resultados e finalizar o modelo de IA para previsão de produtividade agrícola.